In [1]:
import torch

d:\AI_Projects\Ultimate_PyTorch\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
class LinReg:
    def __init__(self,w,b):
        self.w = w
        self.b = b

    def Predict(self,x):
        return self.w*x + self.b

In [3]:
class MAELoss:
    def loss(self, prediction, target):
        error = prediction - target
        return error.abs().mean()


class MSELoss:
    def loss(self, prediction, target):
        error = prediction - target
        return (error ** 2).mean()

mae = MAELoss()
mse = MSELoss()

In [4]:
# Define Model
model = LinReg(w=torch.rand(1,requires_grad=True),b=torch.rand(1,requires_grad=True))
print(model.w,model.b)
model.Predict(5)

tensor([0.0919], requires_grad=True) tensor([0.0454], requires_grad=True)


tensor([0.5048], grad_fn=<AddBackward0>)

In [5]:
x = torch.tensor([1,2,3,4,5,6,7,8,9,10],dtype=torch.float32)
y = torch.tensor([8,13,18,23,28,33,38,43,48,53],dtype=torch.float32)

# Eq: Y = 5*x + 3
y_pred = model.Predict(x)
y_pred

tensor([0.1373, 0.2292, 0.3211, 0.4129, 0.5048, 0.5967, 0.6886, 0.7805, 0.8724,
        0.9643], grad_fn=<AddBackward0>)

In [6]:
error = y_pred -y
error

tensor([ -7.8627, -12.7708, -17.6789, -22.5871, -27.4952, -32.4033, -37.3114,
        -42.2195, -47.1276, -52.0357], grad_fn=<SubBackward0>)

In [7]:
mae.loss(y_pred,y), mse.loss(y_pred,y)

(tensor(29.9492, grad_fn=<MeanBackward0>),
 tensor(1095.6936, grad_fn=<MeanBackward0>))

In [8]:
# Now we have X,Y Pred and loss function, so now we can calculate gradients

# This is for single value scalar
# dL/dw = 2*(w*x + b - y)*x
# dL/db = 2*(w*x + b - y)

# and what we have is vector
dw = (2 * error * x).mean()
db = (2 * error).mean()

dw,db

(tensor(-410.4251, grad_fn=<MeanBackward0>),
 tensor(-59.8984, grad_fn=<MeanBackward0>))

In [9]:
# Applying SGD
lr = 0.01
# SGD Update
model.w = model.w - lr * dw
model.b = model.b - lr * db

In [10]:
# Now do this entire process in Loop: Predict, calculate loss, update weights

for epoch in range(10_000):
    y_pred = model.Predict(x)

    loss = mae.loss(y_pred,y)
    error = y_pred - y

    dw = (2 * error * x).mean()
    db = (2 * error).mean()

    model.w = model.w - lr * dw
    model.b = model.b - lr * db

    if epoch%500==0:
        print(f"Loss: {loss}, W: {model.w} B: {model.b}")


Loss: 6.776846408843994, W: tensor([5.0742], grad_fn=<SubBackward0>) B: tensor([0.7799], grad_fn=<SubBackward0>)
Loss: 0.10351066291332245, W: tensor([5.0383], grad_fn=<SubBackward0>) B: tensor([2.7335], grad_fn=<SubBackward0>)
Loss: 0.012621688656508923, W: tensor([5.0047], grad_fn=<SubBackward0>) B: tensor([2.9675], grad_fn=<SubBackward0>)
Loss: 0.0015393256908282638, W: tensor([5.0006], grad_fn=<SubBackward0>) B: tensor([2.9960], grad_fn=<SubBackward0>)
Loss: 0.00018777846707962453, W: tensor([5.0001], grad_fn=<SubBackward0>) B: tensor([2.9995], grad_fn=<SubBackward0>)
Loss: 2.47955322265625e-05, W: tensor([5.0000], grad_fn=<SubBackward0>) B: tensor([2.9999], grad_fn=<SubBackward0>)
Loss: 1.3256072634248994e-05, W: tensor([5.0000], grad_fn=<SubBackward0>) B: tensor([3.0000], grad_fn=<SubBackward0>)
Loss: 1.3256072634248994e-05, W: tensor([5.0000], grad_fn=<SubBackward0>) B: tensor([3.0000], grad_fn=<SubBackward0>)
Loss: 1.3256072634248994e-05, W: tensor([5.0000], grad_fn=<SubBackwar

In [38]:
# Lets re-start

# Create dataset
x = torch.tensor([1,2,3,4,5,6,7,8,9,10],dtype=torch.float32)
y = 5*x + 3  # Equation is: 5X + 3

# Define Model Architecture:
class LinReg:
    def __init__(self):
        self.w = torch.rand(1,requires_grad=True)
        self.b = torch.rand(1,requires_grad=True)

    def __call__(self,x):
        return self.w*x + self.b

# Define Loss Fun
class MAELoss:
    def __call__(self, prediction, target):
        error = prediction - target
        return error.abs().mean()


class MSELoss:
    def __call__(self, prediction, target):
        error = prediction - target
        return (error ** 2).mean()

mae = MAELoss()
mse = MSELoss()

#Model Instance
model = LinReg()

# Learning Rate
lr = 0.01

# Training Loop
best_loss = torch.inf
patience = 100
counter = 0
min_delta = 1e-7

for epoch in range(2500):

    y_pred = model(x)

    loss = mse(y_pred, y)
    error = y_pred - y

    # Manual SGD
    dw = (2 * error * x).mean()
    db = (2 * error).mean()

    model.w = model.w - lr * dw
    model.b = model.b - lr * db

    # Early Stopping
    current_loss = loss.item()

    if best_loss - current_loss > min_delta:
        best_loss = current_loss
        counter = 0
    else:
        counter += 1

    if epoch % 10 == 0:
        print(
            f"Epoch: {epoch}, "
            f"Loss: {current_loss:.6f}, "
            f"W: {model.w.item():.4f}, "
            f"B: {model.b.item():.4f}"
        )

    if counter >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break

Epoch: 0, Loss: 944.180664, W: 4.2841, B: 0.6031
Epoch: 10, Loss: 1.003748, W: 5.3096, B: 0.8449
Epoch: 20, Loss: 0.922723, W: 5.2968, B: 0.9337
Epoch: 30, Loss: 0.848238, W: 5.2846, B: 1.0189
Epoch: 40, Loss: 0.779767, W: 5.2728, B: 1.1005
Epoch: 50, Loss: 0.716821, W: 5.2616, B: 1.1788
Epoch: 60, Loss: 0.658957, W: 5.2508, B: 1.2539
Epoch: 70, Loss: 0.605764, W: 5.2405, B: 1.3258
Epoch: 80, Loss: 0.556865, W: 5.2306, B: 1.3948
Epoch: 90, Loss: 0.511913, W: 5.2211, B: 1.4610
Epoch: 100, Loss: 0.470591, W: 5.2120, B: 1.5244
Epoch: 110, Loss: 0.432602, W: 5.2032, B: 1.5852
Epoch: 120, Loss: 0.397682, W: 5.1948, B: 1.6435
Epoch: 130, Loss: 0.365579, W: 5.1868, B: 1.6994
Epoch: 140, Loss: 0.336069, W: 5.1791, B: 1.7530
Epoch: 150, Loss: 0.308940, W: 5.1717, B: 1.8044
Epoch: 160, Loss: 0.284002, W: 5.1647, B: 1.8537
Epoch: 170, Loss: 0.261076, W: 5.1579, B: 1.9009
Epoch: 180, Loss: 0.240002, W: 5.1514, B: 1.9462
Epoch: 190, Loss: 0.220628, W: 5.1451, B: 1.9896
Epoch: 200, Loss: 0.202818, W

In [39]:
model(x[4])

tensor([27.9999], grad_fn=<AddBackward0>)